# More on optimal sampling

`imf` can optimally sample mass functions, guaranteeing the ability to construct a cluster that fully utilizes a mass reservoir and follows the shape of a given mass function with no scatter. Here, we show more examples of optimal sampling and provide more information about the algorithm, including some of its subtleties. [Kroupa et al. (2013)](https://doi.org/10.48550/arXiv.1112.3340) and [Kroupa et al. (2026)](https://doi.org/10.48550/arXiv.2410.07311) provide more information about optimal sampling as a concept.

In [ ]:
from imf import Salpeter,Kroupa,ChabrierLogNormal,ChabrierPowerLaw,KoenConvolvedPowerLaw
from imf import make_cluster
import numpy as np
import matplotlib.pyplot as plt
from time import perf_counter

The displays here will all be histograms, for simplicity; here, we'll set up a careful binning procedure that returns a histogram consistent with $dn/dm$.

In [ ]:
def normalization(max_star,massfunc):
    return 1/massfunc.integrate(max_star,massfunc.mmax)[0]

def hist_props(stars,massfunc):
    edges = [stars[0]*1.01]
    for i in range(1,np.floor_divide(len(stars),10)):
        try:
            edge = (stars[10*i-1]+stars[10*i])/2
            edges.append(edge)
        except(IndexError):
            continue
    edges.append(massfunc.mmin)
    edges = np.array(edges)[::-1]
    dm = edges[1:]-edges[:-1]

    weights = np.ones(len(stars))
    for i in range(len(dm)):
        weights[np.logical_and(stars > edges[i],stars < edges[i+1])] *= 1/dm[i]
        
    return edges,weights

Optimal sampling can be invoked through the `'sampling'` keyword in `imf`'s `make_cluster` function. Here, we'll directly compare its output to a randomly sampled cluster.

In [ ]:
opt_cl = make_cluster(1e3,'kroupa',sampling='optimal')
rand_cl = make_cluster(1e3,'kroupa')

In [ ]:
plt.figure()
massfunc = Kroupa()
k = normalization(opt_cl[0],massfunc)
plt.plot(opt_cl,k*massfunc(opt_cl),label='IMF')

edges,weights = hist_props(opt_cl,massfunc)
n,bins,patches = plt.hist(opt_cl,bins=edges,weights=weights,histtype='step',
                          label='optimal',zorder=-2)

rand_cl.sort()
rand_cl = rand_cl[::-1]
edges,weights = hist_props(rand_cl,massfunc)
n,bins,patches = plt.hist(rand_cl,bins=edges,weights=weights,histtype='step',
                          label='random (nearest)',zorder=-1)

plt.title('Random vs. Optimally Sampled Kroupa IMF')
plt.xscale('log'); plt.yscale('log')
plt.xlabel('M'); plt.ylabel('dN/dM')
plt.legend()

`imf` implements optimal sampling according to the prescription of
[Schulz et al. (2015)](https://doi.org/10.1051/0004-6361/201425296), which modifies the original algorithm of [Kroupa et al. (2013)](https://doi.org/10.48550/arXiv.1112.3340). The base concept of this algorithm is that the integrals which produce all of the members and mass of a stellar population:

$$
N_{\rm tot} = \int_{m_{\rm min}}^{m_{\rm max}}\psi(m)dm
$$
$$
M_{\rm tot} = \int_{m_{\rm min}}^{m_{\rm max}}m\psi(m)dm
$$

are broken up into sub-integrals between common bounds that produce a 
$dN$ of 1 and a $dM$ of the total mass of the corresponding star. ($\psi(m)$ is scaled appropriately such that the integrals work out.)

$$
N_{\rm tot} = \int_{m_{\rm min}}^{b_n}\psi(m)dm + \int_{b_n}^{b_{n-1}}\psi(m)dm\, + \,... + \int_{b_1}^{m_{\rm max}}\psi(m)dm = 1 + 1 \, + \,...
$$
$$
M_{\rm tot} = \int_{m_{\rm min}}^{b_n}m\psi(m)dm + \int_{b_n}^{b_{n-1}}m\psi(m)dm + ... + \int_{b_1}^{m_{\rm max}}m\psi(m)dm
$$

`imf`'s implementation first finds the most massive star $m_\star$ by iteratively solving
the following system of equations:

$$
\int_{m_\star}^{m_{\rm max}}\psi(m)dm = 1
$$

$$
M_{\rm tot}(m_\star) - m_\star = \int_{m_{\rm min}}^{m_\star}m\psi(m)dm
$$

which ensure that the IMF is scaled appropriately to produce one 
and only one star with mass $m_\star$. From there, it finds successive stars by locating the points on the IMF which will return $\int_{b_i+1}^{b_i}\psi(m)dm=1$.

.. note:: These equations enforce a direct relationship between the mass
of a cluster $M_{\rm cl}$ and the largest possible mass of a constituent star, $m_{\rm max}$, meaning that that assumption is baked in to optimal sampling as formulated here. Further, in this paradigm there is at most one star which can attain this mass.

The stop criterion for optimal sampling is reaching the lowest minimum stellar mass $m_{\rm min}$, so the `stop_criterion` keyword has no significance for optimal sampling. If you want to cut off earlier, you can specify a tolerance larger than $m_{\rm min}$, and optimal sampling will terminate when the available unsampled mass no longer exceeds that tolerance. (The larger of the two values will be used.) You can provide a mass function with an `mmin` of zero. However, you must then specify a mass tolerance with the `tolerance` keyword, or else optimal sampling will never terminate (there can't be any zero-mass stars, so the mass will asymptote to zero but will not reach it).

.. note:: Optimal sampling relies on integration, meaning that the 
cluster depends on the accuracy of integration for the `MassFunction` being used as a base. Consequently, for functions whose implementations are not fully analytic (e.g. `KoenConvolvedPowerLaw`, which relies on interpolation due to not being feasible to implement analytically) the returned cluster will still use the total mass budget and reproduce these mass functions as expected, but may do so with less accuracy.

We now showcase optimal sampling in action, including how it handles some
edge cases.

In [ ]:
def make_hist(stars,massfunc):
    edges,weights = hist_props(stars,massfunc)
    plt.figure()
    k = normalization(stars[0],massfunc)
    plt.plot(stars,k*massfunc(stars),label='IMF')
    n,bins,patches = plt.hist(stars,bins=edges,weights=weights,histtype='step',label='sampled')
    #plt.title('Random vs. Optimally Sampled Kroupa IMF')
    plt.xscale('log'); plt.yscale('log')
    plt.xlabel('M'); plt.ylabel('dN/dM')
    plt.legend()

In [ ]:
M_res = 1000

In [ ]:
sp = Salpeter()
cl = make_cluster(M_res,massfunc=sp,sampling='optimal')
assert M_res-cl.sum() <= sp.mmin
make_hist(cl,sp)

In [ ]:
kr = Kroupa()
cl = make_cluster(M_res,massfunc=kr,sampling='optimal')
assert M_res-cl.sum() <= kr.mmin
make_hist(cl,kr)

In [ ]:
chl = ChabrierLogNormal(mmin=0.03)
cl = make_cluster(M_res,massfunc=chl,sampling='optimal')
assert M_res-cl.sum() <= chl.mmin
make_hist(cl,chl)

In [ ]:
chp = ChabrierPowerLaw(mmin=0.03)
cl = make_cluster(M_res,massfunc=chp,sampling='optimal')
assert M_res-cl.sum() <= chp.mmin
make_hist(cl,chp)

In [ ]:
koc = KoenConvolvedPowerLaw(0.03,120,1.9,0.4)
cl = make_cluster(M_res,massfunc=koc,sampling='optimal')
assert M_res-cl.sum() <= koc.mmin
make_hist(cl,koc)

As mentioned above, using optimal sampling for semi-analytic functions like `KoenConvolvedPowerLaw` can yield slightly incorrect results due to interpolation not fully capturing the function. Optimal sampling will catch this when it occurs, stop if necessary, and return the generated cluster, which will still be representative in most cases. Accuracy of the total cluster mass may be improvable by increasing the number of subdivisions permitted to `scipy.integrate.quad` (which handles the underlying integration). `KoenConvolvedPowerLaw` has a property `quad_sub_limit` which exists to address this, and in general keywords can be passed to `quad` through the built-in `integrate` functions.

In [ ]:
print(koc.quad_sub_limit)
koc.quad_sub_limit = 100
print(koc.quad_sub_limit)

Example of providing a mass function with mmin = 0 and tolerance = 0. Optimal sampling will throw a ValueError if a nonzero (or negative, or non-finite) lower limit is not provided.

In [ ]:
chl = ChabrierLogNormal()
try:
    cl = make_cluster(1e3,massfunc=chl,sampling='optimal')
except(ValueError):
    print('Caught exception due to zero mmin and tolerance.')
tolerance = 0.1
cl = make_cluster(1e3,massfunc=chl,sampling='optimal',tolerance=tolerance)
assert M_res-cl.sum() <= tolerance
make_hist(cl,chl)

.. warning:: Optimal sampling is a fundamentally iterative process, as 
the mass of each star is dependent on the previous star. As such, optimally sampling massive clusters can take a very long time because of the necessary number of iterations.

In [ ]:
import os
skip_slow_cell = (os.environ.get('READTHEDOCS') == 'True' or
                  os.environ.get('IMF_DOCS_SKIP_SLOW') == '1')
if skip_slow_cell:
    print('Skipping long-running optimal sampling example during docs build.')
else:
    chl = ChabrierLogNormal(mmin=0.03,mmax=150)
    t0 = perf_counter()
    cl = make_cluster(M_res*100,massfunc=chl,sampling='optimal')
    tot_time = perf_counter() - t0
    print(f'Total time to make this cluster was {np.round(tot_time)} seconds.')
    assert M_res-cl.sum() <= chl.mmin
    make_hist(cl,chl)